In [6]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
import numpy as np

# 1. ПОДГОТОВКА ДАННЫХ (DataLoaders)
# Укажите путь к папке, где лежат подпапки с вашими классами
DATA_PATH = 'data'

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

full_dataset = datasets.ImageFolder(
    root=DATA_PATH,
    transform=transform
)

# Загрузка и разделение датасета
full_dataset = datasets.ImageFolder(root=DATA_PATH, transform=transform)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# 2. ОПРЕДЕЛЕНИЕ ФУНКЦИЙ (Обучение и Валидация)
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    running_loss = 0.0
    criterion = nn.CrossEntropyLoss()
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    return running_loss / len(loader)

def validate(model, loader, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    criterion = nn.CrossEntropyLoss()
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    val_f1 = f1_score(all_labels, all_preds, average='macro')
    return running_loss / len(loader), val_f1

# 3. ИНИЦИАЛИЗАЦИЯ МОДЕЛИ (Transfer Learning)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_classes = len(full_dataset.classes)

model = models.resnet50(weights='IMAGENET1K_V1')

# Заморозка весов (кроме последнего блока)
for param in model.parameters():
    param.requires_grad = False

# Замена классификатора (FC layer)[cite: 1]
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.fc.in_features, num_classes)
)

# Разморозка layer4 для fine-tuning[cite: 1]
for param in model.layer4.parameters():
    param.requires_grad = True

model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 4. ЦИКЛ ОБУЧЕНИЯ С EARLY STOPPING[cite: 1]
best_val_loss = float('inf')
patience = 5
no_improve = 0

print(f"Начинаем обучение на устройстве: {device}")

for epoch in range(100):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    val_loss, val_f1 = validate(model, val_loader, device)

    print(f'Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, F1-macro={val_f1:.4f}')

    # Логика сохранения лучшей модели[cite: 1]
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pth')
        no_improve = 0
        print("Модель сохранена!")
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f'Early stopping на эпохе {epoch}!')
            break

Начинаем обучение на устройстве: cpu
Epoch 0: train_loss=0.2522, val_loss=48.0223, F1-macro=0.3434
Модель сохранена!
Epoch 1: train_loss=0.0209, val_loss=8.3657, F1-macro=0.5908
Модель сохранена!
Epoch 2: train_loss=0.0041, val_loss=2.4435, F1-macro=0.6800
Модель сохранена!
Epoch 3: train_loss=0.0093, val_loss=0.2044, F1-macro=0.9655
Модель сохранена!
Epoch 4: train_loss=0.0036, val_loss=0.1630, F1-macro=0.9283
Модель сохранена!
Epoch 5: train_loss=0.0062, val_loss=0.2204, F1-macro=0.9655
Epoch 6: train_loss=0.0039, val_loss=0.3080, F1-macro=0.9333
Epoch 7: train_loss=0.0006, val_loss=0.2522, F1-macro=0.9031
Epoch 8: train_loss=0.0062, val_loss=0.1386, F1-macro=0.9655
Модель сохранена!
Epoch 9: train_loss=0.0037, val_loss=0.0470, F1-macro=0.9283
Модель сохранена!
Epoch 10: train_loss=0.0142, val_loss=0.0632, F1-macro=0.9655
Epoch 11: train_loss=0.0035, val_loss=0.1180, F1-macro=0.8880
Epoch 12: train_loss=0.0045, val_loss=0.5229, F1-macro=0.8438
Epoch 13: train_loss=0.0020, val_loss=0.

In [11]:
import torch
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
import numpy as np

# =========================
# 1. ПОДГОТОВКА ДАННЫХ
# =========================

# Структура:
# data/
#    class_0/
#    class_1/

DATA_PATH = 'data'

transform = transforms.Compose([
    transforms.Resize((64, 64)),  # уменьшили размер для XGBoost
    transforms.ToTensor()
])

# Загрузка датасета
dataset = datasets.ImageFolder(
    root=DATA_PATH,
    transform=transform
)

print("Классы:", dataset.classes)
print("Количество изображений:", len(dataset))

# =========================
# 2. ПРЕОБРАЗОВАНИЕ В NUMPY
# =========================

X = []
y = []

for img, label in dataset:
    # Tensor -> numpy -> flatten
    X.append(img.numpy().flatten())
    y.append(label)

X = np.array(X)
y = np.array(y)

print("Размер X:", X.shape)
print("Размер y:", y.shape)

# =========================
# 3. TRAIN / TEST SPLIT
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

# =========================
# 4. GRID SEARCH
# =========================

param_grid = {
    'n_estimators': [100, 300],
    'max_depth': [3, 5],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.8, 1.0]
}

model = XGBClassifier(
    random_state=42,
    eval_metric='logloss'
)

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=2
)

print("Начинаем обучение...")

grid_search.fit(X_train, y_train)

# =========================
# 5. РЕЗУЛЬТАТЫ
# =========================

print("\nЛучшие параметры:")
print(grid_search.best_params_)

print("\nЛучший F1:")
print(grid_search.best_score_)

# =========================
# 6. ПРОВЕРКА НА TEST
# =========================

best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

f1 = f1_score(y_test, y_pred, average='macro')

print("\nF1 на test:")
print(f1)

Классы: ['class_0', 'class_1']
Количество изображений: 198
Размер X: (198, 12288)
Размер y: (198,)
Train size: 158
Test size: 40
Начинаем обучение...
Fitting 5 folds for each of 16 candidates, totalling 80 fits

Лучшие параметры:
{'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'subsample': 1.0}

Лучший F1:
0.6739104710802823

F1 на test:
0.5670995670995671


In [19]:
import torch
import torch.nn as nn
import torchvision.models as models
import optuna

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_classes = 2  # class_0 и class_1

# =========================
# СОЗДАНИЕ МОДЕЛИ
# =========================

def build_model(dropout):

    model = models.resnet18(weights='IMAGENET1K_V1')

    # Заморозка весов
    for param in model.parameters():
        param.requires_grad = False

    # Новый классификатор
    model.fc = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(model.fc.in_features, num_classes)
    )

    # Разморозим последний блок
    for param in model.layer4.parameters():
        param.requires_grad = True

    return model.to(device)

from sklearn.metrics import f1_score

def train_and_eval(model, lr, batch_size, optimizer_name):

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    criterion = nn.CrossEntropyLoss()

    if optimizer_name == 'Adam':
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    # Обучение 1 эпоха
    model.train()

    for inputs, labels in train_loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

    # Валидация
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for inputs, labels in val_loader:

            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)

            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    f1 = f1_score(all_labels, all_preds, average='macro')

    return f1

def objective(trial):

    lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)

    dropout = trial.suggest_float('dropout', 0.1, 0.5)

    batch_size = trial.suggest_categorical(
        'batch_size',
        [16, 32, 64]
    )

    optimizer_name = trial.suggest_categorical(
        'optimizer',
        ['Adam', 'SGD']
    )

    model = build_model(dropout)

    val_f1 = train_and_eval(
        model,
        lr,
        batch_size,
        optimizer_name
    )

    return val_f1


study = optuna.create_study(direction='maximize')

study.optimize(objective, n_trials=30)

print("Best params:", study.best_params)

print("Best F1:", study.best_value)

[I 2026-05-07 10:24:30,732] A new study created in memory with name: no-name-5392d30f-7c8b-4c51-8c35-1ad4c24806c0


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\user/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 44.7M/44.7M [00:06<00:00, 7.04MB/s]
[I 2026-05-07 10:24:45,922] Trial 0 finished with value: 0.9626517273576096 and parameters: {'lr': 0.008869527745605895, 'dropout': 0.3766420008126291, 'batch_size': 16, 'optimizer': 'Adam'}. Best is trial 0 with value: 0.9626517273576096.
[I 2026-05-07 10:24:53,473] Trial 1 finished with value: 0.6576695194206715 and parameters: {'lr': 0.0008811481951062751, 'dropout': 0.3010089902777654, 'batch_size': 32, 'optimizer': 'Adam'}. Best is trial 0 with value: 0.9626517273576096.
[I 2026-05-07 10:25:01,258] Trial 2 finished with value: 0.5396419437340154 and parameters: {'lr': 2.8569082217180838e-05, 'dropout': 0.4380399919138759, 'batch_size': 32, 'optimizer': 'Adam'}. Best is trial 0 with value: 0.9626517273576096.
[I 2026-05-07 10:25:09,155] Trial 3 finished with value: 0.9283154121863799 and parameters: {'lr'

Best params: {'lr': 0.008869527745605895, 'dropout': 0.3766420008126291, 'batch_size': 16, 'optimizer': 'Adam'}
Best F1: 0.9626517273576096
